# Phase 3 : Ingénierie des Caractéristiques (Feature Engineering)

Ce notebook présente la création de descripteurs prédictifs pour nos modèles de séries temporelles.

## Caractéristiques :
1. **Lags temporels** : Retards de 1, 2, 6, et 12 mois.
2. **Moyennes mobiles et volatilité** : Sur 3, 6, et 12 mois.
3. **Taux de croissance annuel (YoY)**.
4. **Composants saisonniers** : Sin/Cos du mois, trimestres, indicateurs de vacances, Ramadan mouvant, etc.
5. **Variables macroéconomiques retardées**.
6. **Indicateurs d'anomalies** : Isolation Forest, Z-score, Prophet.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd

sys.path.append(os.path.abspath('..'))
from src.config import DATA_DIR, TARGET_COL, TRAIN_END_DATE, TEST_START_DATE
import src.features as feat

## 1. Chargement des données

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'merged_tourism_data_final.csv'))
df['Date'] = pd.to_datetime(df['Date'])
df.head()

## 2. Génération des descripteurs

In [ ]:
df_featured = feat.build_features(df)
features_list = feat.get_feature_list()
print(f"Nombre de descripteurs générés : {len(features_list)}")
df_featured[features_list].head()

## 3. Découpage temporel chronologique

In [ ]:
df_ml = df_featured.dropna(subset=[TARGET_COL]).copy()
valid_features = [c for c in features_list if c in df_ml.columns]

X_train = df_ml[df_ml['Date'] <= TRAIN_END_DATE][valid_features]
y_train = df_ml[df_ml['Date'] <= TRAIN_END_DATE][TARGET_COL]
X_test = df_ml[df_ml['Date'] >= TEST_START_DATE][valid_features]
y_test = df_ml[df_ml['Date'] >= TEST_START_DATE][TARGET_COL]

print(f"Entraînement : {X_train.shape[0]} mois")
print(f"Test : {X_test.shape[0]} mois")

## 4. Sauvegarde des splits

In [ ]:
separated_dir = os.path.join(DATA_DIR, 'separted')
os.makedirs(separated_dir, exist_ok=True)

X_train.to_csv(os.path.join(separated_dir, 'X_train.csv'), index=False)
y_train.to_csv(os.path.join(separated_dir, 'y_train.csv'), index=False)
X_test.to_csv(os.path.join(separated_dir, 'X_test.csv'), index=False)
y_test.to_csv(os.path.join(separated_dir, 'y_test.csv'), index=False)

print("Sauvegarde des splits terminée !")